# Warehouse API

Every `ecg-warehouse` CLI scenario from a notebook. `WarehouseAPI` runs the same code as the CLI, but returns
DataFrames and dicts instead of printing them.

| CLI | Notebook |
| --- | --- |
| `ecg-warehouse --config cfg/local.json ...` | `wh = WarehouseAPI("cfg/local.json")` |
| `sql "<query>"` | `wh.sql("<query>")` |
| `ingest ptb-xl <dir> --rate 500 [--batch-size N]` | `wh.ingest_ptb_xl(<dir>, rate=500[, batch_size=N])` |
| `dataset build <spec.json> [--mode, --dev-tag, --release-type, --store-root, --s3-*]` | `wh.build_dataset(<spec>[, mode=, dev_tag=, release_type=, store_root=, s3_bucket=, ...])` |
| `dataset list` | `wh.list_datasets()` |
| `dataset export <name>@<version> --format csv\|npz --output <dir> [--split, --limit]` | `wh.export_dataset(<ref>, format=, output=[, split=, limit=])` |

Ingests clean up the signals store afterwards, as the CLI does (`cleanup=False` skips it). Errors are raised as
exceptions.

In [ ]:
import os
from pathlib import Path

import numpy as np

from ecg_warehouse.api import WarehouseAPI

# Warehouse paths (data/, cfg/) are relative to the repository root, and Jupyter may start in notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

wh = WarehouseAPI()  # default local warehouse; WarehouseAPI("cfg/local.json") is `--config cfg/local.json`

## SQL

`recordings` and `diagnoses` are registered in DuckDB. Each `wh.sql` call scans both tables; keep a connection from
`wh.duckdb_connection()` for a series of queries.

In [ ]:
wh.sql("SELECT source, sampling_rate_hz, count(*) AS recordings FROM recordings GROUP BY ALL ORDER BY ALL")

In [ ]:
conn = wh.duckdb_connection()
conn.sql("SELECT code_system, code, count(*) AS n FROM diagnoses GROUP BY ALL ORDER BY n DESC LIMIT 20").df()

## Ingest

Set a directory to run an ingest. PTB-XL skips already ingested records.

In [ ]:
PTB_XL_DIR = None  # e.g. "/data/ptb-xl-1.0.3"
PTB_XL_RATE = 500  # 100 or 500

if PTB_XL_DIR is not None:
    wh.ingest_ptb_xl(PTB_XL_DIR, rate=PTB_XL_RATE)

## Build a dataset

`spec` is a spec file path, a dict or a `DatasetSpec`. Without `mode` or `release_type` this is a dev build (a
release build when `CI` is set); `release_type="major"|"minor"|"patch"` makes a release.

In [ ]:
SPEC = "cfg/datasets/afib_binary.json"

manifest = wh.build_dataset(SPEC)
DATASET_REF = f"{manifest['name']}@{manifest['version']}"
DATASET_REF

## List datasets

In [ ]:
wh.list_datasets()[["name", "version", "mode", "release_type", "recording_count", "diagnosis_count", "manifest_uri"]]

## Export

`csv` writes `recordings.csv` and `diagnoses.csv`; `npz` writes `signals.npz`, `meta.csv` and `diagnoses.csv` and
needs a dataset built with `"signals_format": "icechunk"` whose signals share one shape.

In [ ]:
EXPORT_DIR = Path("tmp/export") / DATASET_REF

wh.export_dataset(DATASET_REF, format="csv", output=EXPORT_DIR)

In [ ]:
wh.export_dataset(DATASET_REF, format="npz", output=EXPORT_DIR, split="train", limit=100)

In [ ]:
data = np.load(EXPORT_DIR / "signals.npz")
{key: data[key].shape for key in data.files}